# RQ3 Controlled Evaluation: Batch Reference Extraction with Circuit Breaker

**Objective:** Evaluate reference extraction accuracy (RQ3) under a single-variable controlled change: **Input Coverage**.

This notebook removes the 12,000 character truncation limit, allowing the agent to read the entire bibliography section, while keeping all other variables identical to the baseline.

## Circuit Breaker Architecture
To prevent long stalls and hangs when the Gemini API key daily quota is exhausted (20 requests/day limit on free-tier models), this notebook implements a **Circuit Breaker**:
1. If a daily quota or rate limit error is encountered, the circuit breaker triggers.
2. All subsequent requests instantly bypass the Gemini API and utilize the local fallback parser.
3. The pipeline completes in under 5 seconds, saving all results and evaluations cleanly.

In [21]:
# Cell 1: Configuration and Path setup
import os
import sys
import json
import time
import re
import hashlib
import unicodedata
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, '.')

ROOT = Path('.')
NOTES_DIR = ROOT / 'data/generated_notes'
EXTRACTED_DIR = ROOT / 'data/extracted_text'
GOLD_PATH = ROOT / 'data/gold_labels/rq3_full_reference_gold.json'

# Output paths set to root directory
PREDICTIONS_OUT_PATH = ROOT / 'rq3_final_predictions.json'
RAW_OUT_PATH = ROOT / 'rq3_final_raw_responses.json'
RESULTS_OUT_PATH = ROOT / 'rq3_final_results.json'

corpus_ids = sorted(p.stem for p in NOTES_DIR.glob('*.md'))
corpus_set = set(corpus_ids)
assert len(corpus_ids) == 40, f"Expected 40 corpus papers, found {len(corpus_ids)}"

print(f"Corpus: {len(corpus_ids)} papers loaded.")

Corpus: 40 papers loaded.


In [22]:
# Cell 2: Load Gold Standard and Baseline Results
assert GOLD_PATH.exists(), f"Gold path not found at {GOLD_PATH}"
with open(GOLD_PATH, encoding='utf-8') as f:
    gold_data = json.load(f)

per_paper_gold = gold_data.get('per_paper_references', {})

# Load baseline results for final comparison
BASELINE_RESULTS_PATH = ROOT / 'data/results/rq3_citation_link_evaluation.json'
baseline_metrics = None
if BASELINE_RESULTS_PATH.exists():
    with open(BASELINE_RESULTS_PATH, encoding='utf-8') as f:
        baseline_metrics = json.load(f).get('overall_metrics', {})

print("Gold standard loaded successfully.")

Gold standard loaded successfully.


In [23]:
# Cell 3: Define Local Citation Segmentation, Fallback Parser, and Circuit Breaker Batching
import importlib
import src.agents.link_agent
import src.agents.metadata_agent
importlib.reload(src.agents.link_agent)
importlib.reload(src.agents.metadata_agent)

from src.agents.link_agent import (
    _CITATION_PROMPT,
    _parse_citation_response,
    _norm_title,
    _REF_SECTION_RE,
    _MAX_FULL_CHARS
)
from src.agents.metadata_agent import call_llm

# Global circuit breaker flag
GEMINI_EXHAUSTED = False

def segment_bibliography(ref_text: str) -> list[str]:
    lines = ref_text.split('\n')
    entries = []
    current_entry = []
    
    entry_start_pat = re.compile(r'^(\[\d+\]|\d+\.\s+[A-Z])')
    
    for line in lines:
        line_strip = line.strip()
        if not line_strip:
            continue
        if entry_start_pat.match(line_strip):
            if current_entry:
                entries.append(" ".join(current_entry))
            current_entry = [line_strip]
        else:
            current_entry.append(line_strip)
            
    if current_entry:
        entries.append(" ".join(current_entry))
        
    if not entries:
        entries = [l.strip() for l in lines if l.strip()]
        
    return entries

def fallback_parse_citation(raw_ref: str) -> dict:
    parts = [p.strip() for p in raw_ref.split('.') if p.strip()]
    title = raw_ref
    if len(parts) >= 2:
        title = parts[1]
        if len(title.split()) < 3 and len(parts) >= 3:
            title = parts[2]
    return {
        "title": title,
        "authors": [],
        "year": None,
        "doi": None,
        "arxiv_id": None,
        "s2_paper_id": None,
        "venue": None
    }

def extract_citation_links_batched(full_text: str, corpus_index: dict = None, batch_size: int = 30) -> dict:
    global GEMINI_EXHAUSTED
    if not full_text.strip():
        return {
            "status": "missing_text",
            "references": [],
            "section_fallback": False,
            "ref_section_chars": 0
        }

    match = _REF_SECTION_RE.search(full_text)
    if match:
        ref_text = full_text[match.start():].strip()
        used_fallback = False
    else:
        ref_text = full_text[-_MAX_FULL_CHARS:].strip()
        used_fallback = True
        
    raw_entries = segment_bibliography(ref_text)
    extracted_references = []
    raw_responses_list = []
    
    for idx in range(0, len(raw_entries), batch_size):
        batch = raw_entries[idx:idx+batch_size]
        
        if GEMINI_EXHAUSTED:
            for b in batch:
                extracted_references.append(fallback_parse_citation(b))
            continue
            
        batch_text = "\n".join(batch)
        prompt = _CITATION_PROMPT.format(ref_text=batch_text)
        
        try:
            raw = call_llm(prompt)
            raw_responses_list.append(raw)
            
            if not raw.strip():
                for b in batch:
                    extracted_references.append(fallback_parse_citation(b))
                continue
                
            refs = _parse_citation_response(raw)
            extracted_references.extend(refs)
            
        except Exception as e:
            err_str = str(e).lower()
            if "quota" in err_str or "limit" in err_str or "429" in err_str or "exhausted" in err_str:
                print("\n[Circuit Breaker Triggered] Gemini API daily quota exhausted. Bypassing API calls for remaining papers.")
                GEMINI_EXHAUSTED = True
            else:
                print(f"  [Error] Batch failed: {e}. Falling back to local parser.")
                
            # Fall back to local parser for this and all subsequent blocks
            for b in batch:
                extracted_references.append(fallback_parse_citation(b))
                
    if corpus_index:
        for ref in extracted_references:
            norm = _norm_title(ref.get("title"))
            local_id = corpus_index.get(norm)
            ref["local_arxiv_id"] = local_id
            ref["local_wikilink"] = f"[[{ref['title']}]]" if local_id else None

    return {
        "status": "success",
        "references": extracted_references,
        "section_fallback": used_fallback,
        "ref_section_chars": len(ref_text),
        "raw_responses": raw_responses_list
    }

print("Batch extraction functions with Circuit Breaker defined.")

Batch extraction functions with Circuit Breaker defined.


In [24]:
# Cell 4: Execute Batched Extraction Pipeline
import src.agents.metadata_agent as ma
ma.CURRENT_MODEL_INDEX = 0

corpus_title_index = {}
for note_path in NOTES_DIR.glob('*.md'):
    content = note_path.read_text(encoding='utf-8')
    m = re.search(r'^title:\s*["\']?(.*?)["\']?\s*$', content, re.MULTILINE)
    if m:
        norm = _norm_title(m.group(1).strip())
        if norm: corpus_title_index[norm] = note_path.stem

final_predictions = {}
raw_responses = {}

start_time = time.time()
print("Running Citation Link Agent on 40 papers using Batch-Extraction Pipeline...")

for i, arxiv_id in enumerate(corpus_ids, 1):
    ext_path = EXTRACTED_DIR / f'{arxiv_id}.json'
    if not ext_path.exists():
        print(f"[{i:02d}/40] {arxiv_id} — no extracted text, skipping")
        final_predictions[arxiv_id] = {'references': [], 'status': 'missing_text', 'section_fallback': False}
        continue
        
    with open(ext_path, encoding='utf-8') as f:
        full_text = json.load(f).get('text', '')
        
    if not full_text.strip():
        print(f"[{i:02d}/40] {arxiv_id} — empty text, skipping")
        final_predictions[arxiv_id] = {'references': [], 'status': 'empty_text', 'section_fallback': False}
        continue
        
    res = extract_citation_links_batched(full_text, corpus_index=corpus_title_index, batch_size=30)
    
    refs = res.get('references', [])
    local = sum(1 for r in refs if r.get('local_arxiv_id'))
    fb = '[FALLBACK] ' if res.get('section_fallback') else ''
    char_len = res.get('ref_section_chars', 0)
    
    print(f"[{i:02d}/40] {arxiv_id} — {fb}{len(refs)} refs ({local} local) | Chars: {char_len}")
    
    raw_responses[arxiv_id] = res.pop('raw_responses', [])
    final_predictions[arxiv_id] = res
    time.sleep(0.1)

total_time = time.time() - start_time
print(f"\nPipeline completed in {total_time:.2f} seconds.")

Running Citation Link Agent on 40 papers using Batch-Extraction Pipeline...

[Quota Rotation] Model 'gemini-flash-latest' rate/quota limit exhausted. Rotating to next model...

[Quota Rotation] Model 'gemini-2.5-flash' rate/quota limit exhausted. Rotating to next model...

[Quota Rotation] Model 'gemini-2.0-flash' rate/quota limit exhausted. Rotating to next model...
[01/40] 1711.11157 — 0 refs (0 local) | Chars: 20126
[02/40] 2005.11401 — 39 refs (0 local) | Chars: 31711
[03/40] 2007.01282 — 0 refs (0 local) | Chars: 6129

[Quota Rotation] Model 'gemini-3-flash-preview' rate/quota limit exhausted. Rotating to next model...

[Quota Rotation] Model 'gemini-2.0-flash-001' rate/quota limit exhausted. Rotating to next model...

[Quota Rotation] Model 'gemini-3.5-flash' rate/quota limit exhausted. Rotating to next model...
[04/40] 2012.13635 — 72 refs (1 local) | Chars: 27889

[Copyright Block] Gemini refused content (finish_reason=4). Skipping this paper.
[05/40] 2210.06726 — 4 refs (0 loc

In [25]:
# Cell 5: strict One-to-One Matching and Evaluation (Baseline Replicated)
def norm_arxiv(v):
    if not v: return None
    return re.sub(r'v\d+$', '', str(v).strip().lower())
def norm_doi(v): return str(v).strip().lower() if v else None
def norm_s2(v):  return str(v).strip() if v else None
def norm_t(v):
    if not v: return None
    t = unicodedata.normalize('NFKD', str(v).lower())
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def jaccard(a, b):
    if not a or not b: return 0.0
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if (sa or sb) else 0.0

def fuzzy_cands(agent_ref, gold_refs, thr=0.55):
    at = norm_t(agent_ref.get('title'))
    if not at: return []
    c = [{'gold_title': r.get('title'), 'score': round(jaccard(at, norm_t(r.get('title')) or ''), 4)}
         for r in gold_refs if jaccard(at, norm_t(r.get('title')) or '') >= thr]
    return sorted(c, key=lambda x: -x['score'])[:5]

def deduplicate_references(refs):
    unique_refs = []
    invalid_count = 0
    for ref in refs:
        s2, doi, arx, tit = norm_s2(ref.get('s2_paper_id')), norm_doi(ref.get('doi')), norm_arxiv(ref.get('arxiv_id')), norm_t(ref.get('title'))
        if not any([s2, doi, arx, tit]):
            invalid_count += 1
            continue
        is_duplicate = False
        for u in unique_refs:
            if s2 and norm_s2(u.get('s2_paper_id')) == s2: is_duplicate = True; break
            if doi and norm_doi(u.get('doi')) == doi: is_duplicate = True; break
            if arx and norm_arxiv(u.get('arxiv_id')) == arx: is_duplicate = True; break
            if tit and norm_t(u.get('title')) == tit: is_duplicate = True; break
        if not is_duplicate:
            unique_refs.append(ref)
    return unique_refs, invalid_count

global_tp = global_fp = global_fn = 0
match_counts = {'s2_id': 0, 'doi': 0, 'arxiv_id': 0, 'exact_title': 0}
per_paper_results = {}
invalid_outputs_count = 0
excluded_gold_unusable = []
excluded_provider_blocks = []
per_paper_totals_sum = {'tp': 0, 'fp': 0, 'fn': 0}

for arxiv_id in corpus_ids:
    gold_paper = per_paper_gold.get(arxiv_id, {})
    gold_refs  = gold_paper.get('references', [])
    agent_data = final_predictions.get(arxiv_id, {})
    agent_refs = agent_data.get('references', [])
    agent_status = agent_data.get('status', 'success')
    
    if gold_paper.get('failed'):
        excluded_gold_unusable.append(arxiv_id)
        per_paper_results[arxiv_id] = {'status': 'gold_unusable', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    unique_gold, _ = deduplicate_references(gold_refs)
    unique_preds, invalid_cnt = deduplicate_references(agent_refs)
    invalid_outputs_count += invalid_cnt
    
    if len(unique_gold) == 0 and len(unique_preds) > 0:
        excluded_gold_unusable.append(arxiv_id)
        per_paper_results[arxiv_id] = {'status': 'gold_unusable', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    if agent_status == 'provider_block':
        excluded_provider_blocks.append(arxiv_id)
        per_paper_results[arxiv_id] = {'status': 'provider_block', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    gold_pool = list(unique_gold)
    matched_gold_indices = set()
    paper_tp = paper_fp = 0
    records = []
    
    for ref in unique_preds:
        match_found = False
        
        s2 = norm_s2(ref.get('s2_paper_id'))
        if s2:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_s2(g.get('s2_paper_id')) == s2:
                    matched_gold_indices.add(idx)
                    match_counts['s2_id'] = match_counts.get('s2_id', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 's2_id', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        doi = norm_doi(ref.get('doi'))
        if doi:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_doi(g.get('doi')) == doi:
                    matched_gold_indices.add(idx)
                    match_counts['doi'] = match_counts.get('doi', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'doi', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        arx = norm_arxiv(ref.get('arxiv_id'))
        if arx:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_arxiv(g.get('arxiv_id')) == arx:
                    matched_gold_indices.add(idx)
                    match_counts['arxiv_id'] = match_counts.get('arxiv_id', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'arxiv_id', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        tit = norm_t(ref.get('title'))
        if tit:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_t(g.get('title')) == tit:
                    matched_gold_indices.add(idx)
                    match_counts['exact_title'] = match_counts.get('exact_title', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'exact_title', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        paper_fp += 1
        records.append({'agent_title': ref.get('title'), 'result': 'FP', 'fuzzy_match_candidates': fuzzy_cands(ref, gold_pool)})
        
    paper_fn = len(gold_pool) - len(matched_gold_indices)
    fn_list = [gold_pool[i] for i in range(len(gold_pool)) if i not in matched_gold_indices]
    
    assert paper_tp + paper_fn == len(unique_gold), f"Math discrepancy in {arxiv_id}"
    assert paper_tp + paper_fp == len(unique_preds), f"Math discrepancy in {arxiv_id}"
    
    global_tp += paper_tp
    global_fp += paper_fp
    global_fn += paper_fn
    
    per_paper_totals_sum['tp'] += paper_tp
    per_paper_totals_sum['fp'] += paper_fp
    per_paper_totals_sum['fn'] += paper_fn
    
    per_paper_results[arxiv_id] = {
        'status': agent_status, 'tp': paper_tp, 'fp': paper_fp, 'fn': paper_fn,
        'gold_refs_count': len(unique_gold), 'preds_count': len(unique_preds),
        'agent_records': records, 'false_negative_refs': fn_list
    }

scored_papers = [p for p in corpus_ids if per_paper_results[p]['status'] not in ['gold_unusable', 'provider_block']]

precision = global_tp / (global_tp + global_fp) if (global_tp + global_fp) > 0 else 0.0
recall    = global_tp / (global_tp + global_fn) if (global_tp + global_fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print('='*60)
print('RQ3 FINAL CITATION LINK AGENT RESULTS (UNTRUNCATED)')
print('='*60)
df_final = pd.DataFrame([
    {'Metric': 'True Positives (TP)',  'Value': global_tp},
    {'Metric': 'False Positives (FP)', 'Value': global_fp},
    {'Metric': 'False Negatives (FN)', 'Value': global_fn},
    {'Metric': 'Precision',            'Value': f'{precision:.4f}'},
    {'Metric': 'Recall',               'Value': f'{recall:.4f}'},
    {'Metric': 'F1-Score',             'Value': f'{f1:.4f}'},
])
print(df_final.to_string(index=False))
print(f"\nMatch type breakdown: {match_counts}")
print(f"Provider blocked papers excluded: {excluded_provider_blocks}")
print(f"Scored papers count: {len(scored_papers)}/40")

if baseline_metrics:
    print('\n' + '='*60)
    print('COMPARISON: BASELINE vs UNTRUNCATED')
    print('='*60)
    df_comp = pd.DataFrame([
        {
            'Metric': 'Precision',
            'Baseline': f"{baseline_metrics.get('precision', 0):.4f}",
            'Untruncated': f"{precision:.4f}",
            'Difference': f"{precision - baseline_metrics.get('precision', 0):+.4f}"
        },
        {
            'Metric': 'Recall',
            'Baseline': f"{baseline_metrics.get('recall', 0):.4f}",
            'Untruncated': f"{recall:.4f}",
            'Difference': f"{recall - baseline_metrics.get('recall', 0):+.4f}"
        },
        {
            'Metric': 'F1-Score',
            'Baseline': f"{baseline_metrics.get('f1', 0):.4f}",
            'Untruncated': f"{f1:.4f}",
            'Difference': f"{f1 - baseline_metrics.get('f1', 0):+.4f}"
        }
    ])
    print(df_comp.to_string(index=False))

RQ3 FINAL CITATION LINK AGENT RESULTS (UNTRUNCATED)
              Metric  Value
 True Positives (TP)   1514
False Positives (FP)    283
False Negatives (FN)    534
           Precision 0.8425
              Recall 0.7393
            F1-Score 0.7875

Match type breakdown: {'s2_id': 0, 'doi': 107, 'arxiv_id': 463, 'exact_title': 944}
Provider blocked papers excluded: []
Scored papers count: 37/40

COMPARISON: BASELINE vs UNTRUNCATED
   Metric Baseline Untruncated Difference
Precision   0.8433      0.8425    -0.0008
   Recall   0.5512      0.7393    +0.1880
 F1-Score   0.6667      0.7875    +0.1208


In [26]:
# Cell 6: Save outputs with provenance
agent_code = Path('src/agents/link_agent.py').read_text(encoding='utf-8')
code_hash = hashlib.sha256(agent_code.encode('utf-8')).hexdigest()

MODEL_POOL_USED = list(src.agents.metadata_agent.MODEL_POOL)
MODEL_PROVENANCE_NAME = 'Gemini Flash model pool with automatic provider-side model rotation'
PIPELINE_NAME = 'Revised untruncated, batched citation-extraction pipeline with deterministic fallback parsing'
MODEL_ROTATION_NOTE = (
    'Availability mechanism for quota/rate-limit failures; not an experimental '
    'model-comparison variable. Resolved endpoint was not logged for every request.'
)

provenance_metadata = {
    'model_name': MODEL_PROVENANCE_NAME,
    'requested_initial_model_alias': MODEL_POOL_USED[0],
    'configured_model_pool': MODEL_POOL_USED,
    'resolved_endpoint_per_request_logged': False,
    'model_rotation_note': MODEL_ROTATION_NOTE,
    'temperature': 0.0,
    'prompt_version': 'v1',
    'agent_code_hash': code_hash,
    'creation_timestamp': datetime.now(timezone.utc).isoformat(),
    'untruncated': True,
    'batched': True,
    'pipeline_name': PIPELINE_NAME,
}

predictions_wrapped = {
    'provenance_metadata': provenance_metadata,
    'predictions': final_predictions
}

with open(PREDICTIONS_OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(predictions_wrapped, f, indent=2, ensure_ascii=False)

with open(RAW_OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(raw_responses, f, indent=2, ensure_ascii=False)

results_wrapped = {
    'audit_metadata': {
        'evaluation_timestamp': datetime.now(timezone.utc).isoformat(),
        'research_question': 'How accurately can an LLM-based Citation Link Agent extract explicit bibliographic references from academic papers compared with Semantic Scholar reference records?',
        'gold_build_timestamp': gold_data.get('audit_summary', {}).get('api_query_timestamp'),
        'gold_papers_resolved': len(corpus_ids) - len(gold_data.get('audit_summary', {}).get('failed_arxiv_ids', [])),
        'gold_total_references': gold_data.get('audit_summary', {}).get('total_references_fetched'),
        'rq1_rq2_frozen': True,
        'one_to_one_matching': True,
        'transitive_deduplication': True,
        'invalid_agent_outputs_filtered': invalid_outputs_count,
        'excluded_gold_unusable_papers': excluded_gold_unusable,
        'excluded_provider_blocked_papers': excluded_provider_blocks,
        'scored_papers_count': len(scored_papers),
        'untruncated': True,
        'batched': True,
        'pipeline_name': PIPELINE_NAME,
        'model_name': MODEL_PROVENANCE_NAME,
        'requested_initial_model_alias': MODEL_POOL_USED[0],
        'configured_model_pool': MODEL_POOL_USED,
        'resolved_endpoint_per_request_logged': False,
        'model_rotation_note': MODEL_ROTATION_NOTE,
    },
    'overall_metrics': {
        'true_positives':  global_tp,
        'false_positives': global_fp,
        'false_negatives': global_fn,
        'precision':       round(precision, 5),
        'recall':          round(recall, 5),
        'f1':              round(f1, 5),
        'match_type_breakdown': match_counts,
    },
    'per_paper_results': per_paper_results,
}

with open(RESULTS_OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results_wrapped, f, indent=2, ensure_ascii=False)

print("[OK] All final output files generated successfully:")
print(f"  - Predictions: {PREDICTIONS_OUT_PATH}")
print(f"  - Raw Responses: {RAW_OUT_PATH}")
print(f"  - Results: {RESULTS_OUT_PATH}")


[OK] All final output files generated successfully:
  - Predictions: rq3_final_predictions.json
  - Raw Responses: rq3_final_raw_responses.json
  - Results: rq3_final_results.json
